# Control flow, functions and first steps with pandas

The [previous block](03-lecture-notebooks-and-data-types.ipynb) introduced values, types and collections. Programs become useful when they make decisions (**conditions**), repeat work (**loops**) and give a rule a name so that it can be reused (**functions**). Python's standard library and third-party **packages** add ready-made tools; the most important one for this course is **pandas**, which turns a list of records into a table that can be filtered, sorted and summarised in a few lines. The block ends with the real Berlin listings and with tips for working with AI coding assistants.

```mermaid
flowchart LR
  C["Conditions<br/>if, elif, else"] --> L["Loops<br/>for, while,<br/>comprehensions"]
  L --> F["Functions<br/>parameters, return,<br/>docstrings"]
  F --> I["Modules and<br/>packages<br/>import"]
  I --> P["pandas<br/>Series, DataFrame"]
  P --> A["Analysis of<br/>12,776 listings"]
```

In [1]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Conditions: `if`, `elif`, `else`

An `if` statement runs a block of code only when a condition is true. The line ends with a colon, and the block below it is **indented** by four spaces; the indentation, not a bracket, marks where the block ends. `elif` ("else if") tests a further condition when the earlier ones were false, and `else` catches everything that is left. Python checks the conditions from top to bottom and runs **only the first** block whose condition is true.

In [2]:
nightly_price = 75
nights = 3
cleaning_fee = 25
budget = 250

total = nightly_price * nights + cleaning_fee
if total <= budget:
    print("within budget")
else:
    print("over budget")

within budget


The total is exactly €250, and `<=` includes the boundary, so the stay is within budget. Boundaries are where conditions go wrong: test 249, 250 and 251 when you write one.

With `elif`, a value falls into exactly one of several bands. Because the conditions are checked in order, each later condition only needs its upper limit:

In [3]:
def describe(price):
    if price is None:
        return "no price"
    elif price < 60:
        return "budget"
    elif price < 150:
        return "mid-range"
    else:
        return "premium"


for price in [None, 45, 60, 149.5, 300]:
    print(price, describe(price))

None no price
45 budget
60 mid-range
149.5 mid-range
300 premium


(The `def` and `for` lines are explained in the next two sections; here they let us try five prices at once.) The test for `None` comes first: `None < 60` would stop with a `TypeError`.

Conditions combine with `and`, `or` and `not`. Values that are not Booleans also count as true or false in a condition: `0`, `""`, `None` and empty collections count as false, everything else as true. Write the comparison explicitly when the difference matters: `if price:` treats a missing price and a price of 0 the same way, `if price is None:` does not.

## Loops: `for`, `while`, comprehensions

A `for` loop takes the items of a collection one at a time and runs its block for each. A typical pattern is the **accumulator**: start with an empty result before the loop and add to it inside the loop.

In [4]:
records = [
    {"district": "Pankow", "nightly_price": 75, "available": True},
    {"district": "Mitte", "nightly_price": 90, "available": False},
    {"district": "Neukölln", "nightly_price": 110, "available": True},
]

candidates = []                      # the accumulator starts empty
for record in records:
    total = record["nightly_price"] * 3 + 25
    if record["available"] and total <= 300:
        candidates.append(record["district"])

print(candidates)

['Pankow']


Pankow costs €250 for three nights; Mitte is not available; Neukölln costs €355 and is over the budget of €300.

Three helpers are used with `for`:

- `range(n)` produces the whole numbers 0, 1, …, n − 1; `range(1, 11)` produces 1 to 10.
- `enumerate(items)` gives each item together with its position.
- `zip(a, b)` pairs the items of two lists position by position.

A loop over a dictionary goes over its keys; `.items()` gives key and value together.

In [5]:
for position, record in enumerate(records):
    print(position, record["district"])

districts = ["Pankow", "Mitte", "Neukölln"]
prices = [75, 90, 110]
for district, price in zip(districts, prices):
    print(f"{district}: €{price}")

for key, value in records[0].items():
    print(key, "=", value)

0 Pankow
1 Mitte
2 Neukölln
Pankow: €75
Mitte: €90
Neukölln: €110
district = Pankow
nightly_price = 75
available = True


A `while` loop repeats as long as its condition is true. It fits when the number of repetitions is not known in advance. Its condition must eventually become false, or the loop never ends (in a notebook, stop it with the ■ button or *Interrupt*). `break` leaves a loop early; `continue` skips to the next item.

In [6]:
# How many nights at €75 (plus a €25 fee) fit into a budget of €400?
nights = 0
while 75 * (nights + 1) + 25 <= 400:
    nights += 1
print(nights)

5


A **comprehension** builds a new list (or dictionary or set) from a loop in one line: `[expression for item in collection if condition]`. Use it for short transformations; write a normal loop when the work needs several steps.

In [7]:
three_night_totals = [r["nightly_price"] * 3 + 25 for r in records]
available = [r["district"] for r in records if r["available"]]
price_by_district = {r["district"]: r["nightly_price"] for r in records}
print(three_night_totals)
print(available)
print(price_by_district)

[250, 295, 355]
['Pankow', 'Neukölln']
{'Pankow': 75, 'Mitte': 90, 'Neukölln': 110}


> 📖 **Read more:** [extras/04-python-control-flow.ipynb](extras/04-python-control-flow.ipynb) (Whirlwind Tour of Python): `if`–`elif`–`else` chains, `for` and `while` loops, `break` and `continue`, and the rarely seen `else` block of a loop.

## Functions: parameters, return values, docstrings

A **function** gives a rule a name. `def` defines it; the names in parentheses are its **parameters**; calling the function with **arguments** runs its body with those values. `return` sends the result back to the caller. A parameter can have a **default value**, which makes the argument optional.

The first line inside the function is the **docstring**: a short text saying what the function does, which `help()` shows. The optional **type hints** (`: float`, `-> float`) document what kind of values the function expects and returns; Python does not check them.

In [8]:
def stay_cost(nightly_price: float, nights: int, cleaning_fee: float = 25) -> float:
    """Return the total cost of a stay; the cleaning fee is paid once."""
    return nightly_price * nights + cleaning_fee


print(stay_cost(75, 3))                    # uses the default fee: 250
print(stay_cost(110, 3, cleaning_fee=40))  # keyword argument: 370
help(stay_cost)

250
370
Help on function stay_cost in module __main__:

stay_cost(nightly_price: float, nights: int, cleaning_fee: float = 25) -> float
    Return the total cost of a stay; the cleaning fee is paid once.



**`return` and `print` are different.** `print` shows a value on the screen and the function then returns `None`; `return` hands the value to the code that called the function, which can store it, compare it or pass it on. A function that only prints cannot be reused in a calculation.

Names created inside a function are **local**: they exist only while the function runs. A function that receives everything it needs as arguments and returns its result can be understood, reused and checked on its own. A first check is an `assert` with a case computed by hand; it stops with an `AssertionError` if the result is wrong:

In [9]:
assert stay_cost(75, 3) == 250
assert stay_cost(80, 0, cleaning_fee=20) == 20     # boundary: zero nights, only the fee

Should a stay of zero nights cost the fee or be refused? The check makes you decide what "correct" means. Session 2 turns such checks into automated tests and shows how a function refuses invalid input with an exception.

> 📖 **Read more:** [extras/05-python-functions.ipynb](extras/05-python-functions.ipynb) (Whirlwind Tour of Python): default and keyword arguments, flexible arguments with `*args` and `**kwargs`, and anonymous `lambda` functions used as sort keys.
>
> 📖 **Read more:** [extras/06-python-errors-and-exceptions.ipynb](extras/06-python-errors-and-exceptions.ipynb) (Whirlwind Tour of Python): reading error messages, catching exceptions with `try` and `except`, and raising your own, which Session 2 develops.

## Importing modules and packages

A **module** is a file of Python code that other code can **import**; a **package** is a folder of modules. Python comes with a large **standard library** (`math`, `statistics`, `pathlib`, `datetime`, `json` and many more). **Third-party packages** such as pandas, NumPy and scikit-learn are installed into the environment by uv; the course installs all of them with `uv sync`.

In [10]:
import math                           # the whole module; use it as math.<name>
from statistics import median         # one name from a module
import pandas as pd                   # a package under a short alias, by convention

print(math.sqrt(16), math.ceil(2.1))
print(median([75, 90, 110, 1000]))
print(pd.__version__)

4.0 3
100.0
3.0.6


Imports go at the top of a notebook or file. A `ModuleNotFoundError` means that the package is not installed in the environment the kernel uses: check the kernel (`.venv`) before installing anything. Do not name your own file `pandas.py` or `math.py`: Python would import your file instead of the library.

> 📖 **Read more:** [extras/11-python-modules-and-packages.ipynb](extras/11-python-modules-and-packages.ipynb) (Whirlwind Tour of Python): the forms of the `import` statement, why `import *` is risky, a tour of the standard library, and installing third-party packages.

## pandas: Series and DataFrame

**pandas** is the standard Python package for tables. Its two main types:

- A **DataFrame** is a table with labelled columns and a row **index**. Each column has one data type (`dtype`).
- A **Series** is one column: values, an index and a dtype.

A list of dictionaries, one per record, becomes a DataFrame directly:

In [11]:
listings = pd.DataFrame(records)
print(listings.shape)        # (rows, columns)
print(listings.dtypes)       # one dtype per column
listings

(3, 3)
district           str
nightly_price    int64
available         bool
dtype: object


,district,nightly_price,available
0,Pankow,75,True
1,Mitte,90,False
2,Neukölln,110,True


What pandas adds to a list of dictionaries: every operation applies to a whole column at once (**vectorised**), without a Python loop, and missing values, types and labels are handled consistently. The loop of the section on loops becomes one line:

In [12]:
total = listings["nightly_price"] * 3 + 25           # a Series: one total per row
listings.loc[listings["available"] & (total <= 300), "district"]

0    Pankow
Name: district, dtype: str

## Reading, selecting, filtering, sorting and grouping a table

The course's Berlin listings are a Parquet file prepared by `uv run python case-study/prepare_airbnb.py` (see the [session README](../README.md#before-the-session-and-further-reading)). **Before calculating, define what one row is**: here one listing in the snapshot of 26 June 2026, not a booking, a host or a flat.

In [13]:
real = pd.read_parquet("case-study/data/airbnb/listings.parquet")
print(real.shape)
real[["district", "room_type", "price", "minimum_nights", "number_of_reviews"]].head()

(12776, 42)


,district,room_type,price,minimum_nights,number_of_reviews
0,Pankow,Entire home/apt,160.71,2.0,150
1,Pankow,Entire home/apt,193.33,6.0,7
2,Pankow,Entire home/apt,NaN,180.0,26
3,Friedrichshain-Kreuzberg,Entire home/apt,10.88,93.0,298
4,Pankow,Entire home/apt,372.67,3.0,145


**Selecting.** `df["col"]` selects one column as a Series; `df[["a", "b"]]` (a list of names) selects a smaller DataFrame. `.loc[rows, columns]` selects by **labels**, `.iloc[rows, columns]` by **positions**.

**Filtering.** A comparison on a column gives one `True` or `False` per row, a **Boolean mask**; `df[mask]` or `df.loc[mask, columns]` keeps the rows where it is true. Combine masks with `&` (and), `|` (or) and `~` (not), with parentheses around each comparison: Python's `and` and `or` work on single values, not on whole columns.

In [14]:
print(real.loc[0, "district"], real.iloc[0, 0])           # label-based, then position-based (first column: id)

in_pankow = real["district"] == "Pankow"
print(in_pankow.sum(), "listings in Pankow")          # True counts as 1

short_stay = real["price"].notna() & (real["minimum_nights"] < 28)
short = real.loc[short_stay]
print(len(short), "short-stay listings with a price")

Pankow 3176
1950 listings in Pankow
6701 short-stay listings with a price


**Missing values.** pandas marks a missing number as `NaN`. Find missing values with `.isna()` and `.notna()`, never with `== None`. `count()` counts the values that are present, `len()` counts rows; the difference is the number of missing values. About a third of the listings have no price, and many can only be booked for months: the course compares prices only for **short stays**, a price present and a minimum stay below 28 nights.

In [15]:
print(len(real), real["price"].count(), real["price"].isna().mean().round(3))

12776 8441 0.339


**Sorting.** `sort_values("col")` sorts the rows by a column, `ascending=False` from the largest; `.head(n)` keeps the first rows.

In [16]:
short.sort_values("price", ascending=False)[["district", "room_type", "price"]].head(3)

,district,room_type,price
8434,Steglitz - Zehlendorf,Entire home/apt,10025.0
12710,Spandau,Entire home/apt,7999.2
9515,Treptow - Köpenick,Entire home/apt,4458.0


**Counting and grouping.** `value_counts()` counts the categories of a column. `groupby` splits the rows into groups, applies a summary to each group and combines the results into one table (*split–apply–combine*). Select the column before the summary.

In [17]:
print(real["room_type"].value_counts())
short.groupby("room_type")["price"].agg(["size", "median", "mean"]).round(1)

room_type
Entire home/apt    8846
Private room       3754
Hotel room           89
Shared room          87
Name: count, dtype: int64


,size,median,mean
room_type,,,
Entire home/apt,4688,185.2,226.5
Hotel room,76,174.0,203.1
Private room,1857,97.3,116.2
Shared room,80,49.2,58.8


The **mean** is the sum divided by the count; the **median** is the middle value after sorting. A few very expensive listings pull the mean up but barely move the median: with the prices 75, 90, 110 and 1,000 the mean is 318.75 and the median 100. For skewed data such as prices, report the median, and always say which rows were counted: "median price per night of short-stay listings with a price" is a claim that can be checked, "the price in Berlin" is not.

![Price distributions of short-stay listings by room type, with mean and median marked](../figures/price_by_room_type.png)

![Listing counts and median short-stay prices by district](../figures/listings_by_district.png)

A median of a snapshot describes what was asked for in June 2026. Predicting the price of a *new* listing is a different claim, which needs a model and an evaluation on listings it has not seen; Sessions 6 and 7 build both.

> 📖 **Read more:** [extras/07-pandas-introduction.ipynb](extras/07-pandas-introduction.ipynb) (Microsoft Data Science for Beginners): a beginner walk-through of Series and DataFrames; some examples need updating for current pandas.
>
> 📖 **Read more:** [extras/08-pandas-indexing-and-selection.ipynb](extras/08-pandas-indexing-and-selection.ipynb) (Python Data Science Handbook): why `.loc` and `.iloc` exist, slicing, masking and fancy indexing.
>
> 📖 **Read more:** [extras/10-pandas-groupby.ipynb](extras/10-pandas-groupby.ipynb) (Python Data Science Handbook): split–apply–combine in depth: `aggregate`, `filter`, `transform` and `apply`.
>
> 📖 **Read more:** [extras/09-pandas-merge-and-join.ipynb](extras/09-pandas-merge-and-join.ipynb) (Python Data Science Handbook): combining tables with `pd.merge`, which prepares the joins of Session 3.

## Tips for using AI coding assistants

AI assistants are allowed in this course, and many data teams use them every day. They write plausible code quickly, and plausible is not the same as correct: a pandas expression can filter the wrong rows, count hosts instead of listings or drop missing values without any error.

- **Ask precisely.** Name the input (columns and their types), the output you want and the edge cases: "short stays are listings with a price and `minimum_nights < 28`".
- **Check on a small table.** Run the code on three or four rows whose answer you can compute by hand, like the `records` above, before you run it on 12,776 rows.
- **Check the real result.** Compare row counts before and after each filter, and compute one number in a second, independent way (a mask and `len()` instead of `value_counts()`).
- **Understand every line.** If you cannot explain a line, ask the assistant to explain it, or look it up with `help()`. You are responsible for the code you submit.
- **Document the use.** Record which tool you used and for what, as the course requires for the final project (HTW declaration).
- **Learn first.** In the exercises, solve the problem yourself before asking an assistant. Use it to explain an error or an idea, not to replace the practice that builds the skill.

## Check your understanding

1. In an `if`–`elif`–`else` chain, how many blocks can run? Why does the order of the conditions matter?
2. What is the difference between `return` and `print` in a function?
3. When would you use a `while` loop instead of a `for` loop?
4. Why does a pandas filter use `&` and parentheses instead of `and`?
5. `real["price"].count()` and `len(real)` differ. What does the difference tell you, and why do we report the median price of short stays rather than the mean of all prices?

## Further reading

- Python Software Foundation (2026). *The Python Tutorial*, section 4 (more control flow tools) and section 6 (modules). https://docs.python.org/3/tutorial/controlflow.html
- McKinney, W. (2022). *Python for Data Analysis*, 3rd edition, chapters 2–3 and 5 (Python basics; built-in data structures and functions; getting started with pandas). Open access. https://wesmckinney.com/book/
- pandas development team (2026). *Getting started tutorials*, in particular *How do I select a subset of a DataFrame?* and *How to calculate summary statistics*. https://pandas.pydata.org/docs/getting_started/intro_tutorials/
- pandas development team (2026). *Working with missing data*. https://pandas.pydata.org/docs/user_guide/missing_data.html